# Baseline models

This notebook evaluates the benchmark forecasting models on the
cleaned train, validation and test splits. Simple statistical models are also fit directly here.

All baseline models return predictions and ground truth in raw value space.
`ForecastEvaluator` is responsible for transforming predictions into the
required evaluation space and computing the common metrics.

The current available evaluation metrics are:

1. **Cumulative log-change MAE**
2. **MASE**
3. **Pearson Correlation between predictions and target in cumulative log change space**
4. **Relative MAE versus Persistence**
5. **Persistence win rate**

Bootstrap evaluation metrics over the test datset are available and used by default.

The available benchmark models are:

1. **Persistence** — predicts every future horizon using the final target
   value in the context window.
2. **Mean** — predicts every future horizon using the mean target value over
   the context window.
3. **ARIMA** — fits a separate univariate ARIMA model to the one-step log
   changes of each asset and target channel.
4. **VAR** — fits one multivariate VAR model per target channel across all
   assets.
5. **GARCH** — fits a separate GARCH(1,1) model to each asset and target
   channel, with an optional AR(1), constant or zero conditional mean.
6. **ModernTCN** - multiple ablations have been trained in Colab. The
    best version checkpoint (based on validation loss) is called and used to 
    predict. It is the version which takes all OHLCV as input, adds a time
    of day temporal feature and flattens series into batch (so we dont mix
    across asset - huge reduction in parameter count - 121,138 params in total
    including 256 params added for the temporal feature).

In [ ]:
from pathlib import Path
import sys
from time import perf_counter
import pandas as pd
import torch

# Make sure notebook can import from src/
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

from src.data.load_candle_data import load_candle_splits, clean_candle_splits
from src.evaluation.metrics import ForecastEvaluator
from src.models.persistence import PersistenceBaseline
from src.models.mean import MeanBaseline
from src.models.arima import ArimaBaseline
from src.models.var import VarBaseline
from src.models.garch import GarchBaseline
from src.models.modern_tcn import ModernTCNBaseline
from src.utils.config import load_yaml
from src.utils.metric_tables import make_evaluation_table, make_baseline_summary_table
from src.visualization.forecast_plots import plot_forecast_comparison


In [ ]:
DATA_DIR = Path(
    "/Users/vishalruparelia/Library/CloudStorage/"
    "GoogleDrive-vishal@autonomous-fox.ai/"
    "Shared drives/Vishal/data/cached_datasets/"
    "exp-1m-95s-24y/session"
)

CONFIG_PATH = Path("../configs/forecasting.yaml")

BASELINE_CACHE_ROOT = Path(
    "/Users/vishalruparelia/Library/CloudStorage/"
    "GoogleDrive-vishal@autonomous-fox.ai/"
    "My Drive/dissertation"
)

## Load the data and clean

In [ ]:
config = load_yaml(CONFIG_PATH)

train_raw, val_raw, test_raw = load_candle_splits(DATA_DIR)

train, val, test = clean_candle_splits(
    train_raw,
    val_raw,
    test_raw,
)

print("train samples:", len(train["samples"]))
print("val samples:", len(val["samples"]))
print("test samples:", len(test["samples"]))
print("channels:", test["channels"])
print("assets:", len(test["asset_cols"]))
print("stride:", config['forecasting']['stride'])
print("input features:", config['forecasting']['input_channels'])
print("targets:", config['forecasting']['target_channels'])

#Set global Bootstrap params
BOOTSTRAP_N = 10_000
BOOTSTRAP_CONFIDENCE_LEVEL = 0.95
BOOTSTRAP_SEED = 42


## SUMMARY METRICS/FORECAST PLOTS

Detailed metrics per model with bootstrapped errors are below

In [15]:
models_to_display = [
    "persistence",
    "arima",
    "var",
    "garch",
    "modern_tcn",
    "kronos"
]

baseline_summary_table = make_baseline_summary_table(
    models_to_display=models_to_display,
    namespace=globals(),
    channel="close",
)

display(
    baseline_summary_table.style
    .format(
        "{:.6g}",
        na_rep="—",
    )
    .set_caption(
        "Baseline Test-Set Results"
    )
)

Use the function below to plot the forecasts of any model vs the true price. Can compare multiple models.

In [ ]:
fig, axes, selection = plot_forecast_comparison(
    models=["arima","modern_tcn","kronos",],
    namespace=globals(),
    day=None,
    asset="NVDA",
    horizons=[30]
    
)

print(selection)

## Persistence

In [ ]:
RUN = False

persistence_prediction_path = (
    BASELINE_CACHE_ROOT
    / "persistence"
    / "prediction_result.pt"
)

if RUN:
    persistence = PersistenceBaseline.from_config(
        config
    )

    persistence.fit(
        train_split=train,
        val_split=val,
    )

    persistence_result = persistence.predict(
        split=test,
        batch_size=256,
    )

    persistence_prediction_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    torch.save(
        persistence_result,
        persistence_prediction_path,
    )

else:
    persistence_result = torch.load(
        persistence_prediction_path,
        map_location="cpu",
        weights_only=False,
    )

persistence_evaluator = ForecastEvaluator(
    prediction_result=persistence_result,
    train_split=val,
)


# Persistence predicts zero cumulative log change at every horizon,
# so its cumulative-log-change Pearson correlation or IC is undefined.
persistence_undefined_metrics = {
    "cumulative_log_change_pearson_correlation",
    "cumulative_log_change_cross_sectional_pearson_ic",
    "cumulative_log_change_cross_sectional_spearman_rank_ic",
    "cumulative_log_change_temporal_absolute_correlation"

}

persistence_metric_names = [
    metric_name
    for metric_name in persistence_evaluator.available_metrics
    if metric_name not in persistence_undefined_metrics
]


persistence_results = persistence_evaluator.evaluate(
    metrics=persistence_metric_names,
    reduce_dims=(0, 2),
    bootstrap=True,
    n_bootstrap=BOOTSTRAP_N,
    confidence_level=BOOTSTRAP_CONFIDENCE_LEVEL,
    bootstrap_seed=BOOTSTRAP_SEED,
)

persistence_metric_table = make_evaluation_table(
    metric_results=persistence_results,
    horizons=persistence_evaluator.horizons,
    channels=persistence_evaluator.channels,
)


for metric_name in persistence_metric_names:
    metric_display = (
        persistence_metric_table
        .loc[
            persistence_metric_table["metric"]
            == metric_name
        ]
        .set_index(
            [
                "horizon",
                "channel",
            ]
        )[
            [
                "value",
                "ci_lower",
                "ci_upper",
                "bootstrap_mean",
                "bootstrap_std",
            ]
        ]
    )

    display(
        metric_display.style.set_caption(
            metric_name
        )
    )

## Mean

In [ ]:
RUN = False

mean_prediction_path = (
    BASELINE_CACHE_ROOT
    / "mean"
    / "prediction_result.pt"
)

if RUN:
    mean = MeanBaseline.from_config(
        config
    )

    mean.fit(
        train_split=train,
        val_split=val,
    )

    mean_result = mean.predict(
        split=test,
        batch_size=256,
    )

    mean_prediction_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    torch.save(
        mean_result,
        mean_prediction_path,
    )

else:
    mean_result = torch.load(
        mean_prediction_path,
        map_location="cpu",
        weights_only=False,
    )

mean_evaluator = ForecastEvaluator(
    prediction_result=mean_result,
    train_split=train,
)

mean_results = mean_evaluator.evaluate(
    metrics=mean_evaluator.available_metrics,
    reduce_dims=(0, 2),
    bootstrap=True,
    n_bootstrap=BOOTSTRAP_N,
    confidence_level=BOOTSTRAP_CONFIDENCE_LEVEL,
    bootstrap_seed=BOOTSTRAP_SEED,
)

mean_metric_table = make_evaluation_table(
    metric_results=mean_results,
    horizons=mean_evaluator.horizons,
    channels=mean_evaluator.channels,
)

for metric_name in mean_evaluator.available_metrics:
    metric_display = (
        mean_metric_table
        .loc[
            mean_metric_table["metric"]
            == metric_name
        ]
        .set_index(
            [
                "horizon",
                "channel",
            ]
        )[
            [
                "value",
                "ci_lower",
                "ci_upper",
                "bootstrap_mean",
                "bootstrap_std",
            ]
        ]
    )

    display(
        metric_display.style.set_caption(
            metric_name
        )
    )

## ARIMA

In [ ]:
RUN = False

arima_prediction_path = (
    BASELINE_CACHE_ROOT
    / "arima"
    / "prediction_result.pt"
)

if RUN:
    arima = ArimaBaseline.from_config(
        config,
        fit_mode="simple",
        optim_method="powell",
    )

    arima.fit(
        train_split=train,
        val_split=val,
    )

    arima_result = arima.predict(
        split=test,
        batch_size=32,
    )

    arima_prediction_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    torch.save(
        arima_result,
        arima_prediction_path,
    )

else:
    arima_result = torch.load(
        arima_prediction_path,
        map_location="cpu",
        weights_only=False,
    )

arima_evaluator = ForecastEvaluator(
    prediction_result=arima_result,
    train_split=train,
)

arima_results = arima_evaluator.evaluate(
    metrics=arima_evaluator.available_metrics,
    reduce_dims=(0, 2),
    bootstrap=True,
    n_bootstrap=BOOTSTRAP_N,
    confidence_level=BOOTSTRAP_CONFIDENCE_LEVEL,
    bootstrap_seed=BOOTSTRAP_SEED,
)

arima_metric_table = make_evaluation_table(
    metric_results=arima_results,
    horizons=arima_evaluator.horizons,
    channels=arima_evaluator.channels,
)

for metric_name in arima_evaluator.available_metrics:
    metric_display = (
        arima_metric_table
        .loc[
            arima_metric_table["metric"]
            == metric_name
        ]
        .set_index(
            [
                "horizon",
                "channel",
            ]
        )[
            [
                "value",
                "ci_lower",
                "ci_upper",
                "bootstrap_mean",
                "bootstrap_std",
            ]
        ]
    )

    display(
        metric_display.style.set_caption(
            metric_name
        )
    )

## VAR

In [ ]:
RUN = False

var_prediction_path = (
    BASELINE_CACHE_ROOT
    / "var"
    / "prediction_result.pt"
)

if RUN:
    var = VarBaseline.from_config(
        config,
        maxlags=15,
        ic="aic",
        trend="c",
    )

    var.fit(
        train_split=train,
        val_split=val,
    )

    var_result = var.predict(
        split=test,
        batch_size=256,
    )

    var_prediction_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    torch.save(
        var_result,
        var_prediction_path,
    )

else:
    var_result = torch.load(
        var_prediction_path,
        map_location="cpu",
        weights_only=False,
    )

var_evaluator = ForecastEvaluator(
    prediction_result=var_result,
    train_split=train,
)

var_results = var_evaluator.evaluate(
    metrics=var_evaluator.available_metrics,
    reduce_dims=(0, 2),
    bootstrap=True,
    n_bootstrap=BOOTSTRAP_N,
    confidence_level=BOOTSTRAP_CONFIDENCE_LEVEL,
    bootstrap_seed=BOOTSTRAP_SEED,
)

var_metric_table = make_evaluation_table(
    metric_results=var_results,
    horizons=var_evaluator.horizons,
    channels=var_evaluator.channels,
)

for metric_name in var_evaluator.available_metrics:
    metric_display = (
        var_metric_table
        .loc[
            var_metric_table["metric"]
            == metric_name
        ]
        .set_index(
            [
                "horizon",
                "channel",
            ]
        )[
            [
                "value",
                "ci_lower",
                "ci_upper",
                "bootstrap_mean",
                "bootstrap_std",
            ]
        ]
    )

    display(
        metric_display.style.set_caption(
            metric_name
        )
    )

## GARCH

In [ ]:
RUN = False

garch_prediction_path = (
    BASELINE_CACHE_ROOT
    / "garch"
    / "prediction_result.pt"
)

if RUN:
    garch = GarchBaseline.from_config(
        config,
        mean="AR",
        return_scale=10000.0,
    )

    garch.fit(
        train_split=train,
        val_split=val,
    )

    garch_result = garch.predict(
        split=test,
        batch_size=256,
    )

    garch_prediction_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    torch.save(
        garch_result,
        garch_prediction_path,
    )

else:
    garch_result = torch.load(
        garch_prediction_path,
        map_location="cpu",
        weights_only=False,
    )

garch_evaluator = ForecastEvaluator(
    prediction_result=garch_result,
    train_split=train,
)

garch_results = garch_evaluator.evaluate(
    metrics=garch_evaluator.available_metrics,
    reduce_dims=(0, 2),
    bootstrap=True,
    n_bootstrap=BOOTSTRAP_N,
    confidence_level=BOOTSTRAP_CONFIDENCE_LEVEL,
    bootstrap_seed=BOOTSTRAP_SEED,
)

garch_metric_table = make_evaluation_table(
    metric_results=garch_results,
    horizons=garch_evaluator.horizons,
    channels=garch_evaluator.channels,
)

for metric_name in garch_evaluator.available_metrics:
    metric_display = (
        garch_metric_table
        .loc[
            garch_metric_table["metric"]
            == metric_name
        ]
        .set_index(
            [
                "horizon",
                "channel",
            ]
        )[
            [
                "value",
                "ci_lower",
                "ci_upper",
                "bootstrap_mean",
                "bootstrap_std",
            ]
        ]
    )

    display(
        metric_display.style.set_caption(
            metric_name
        )
    )

## ModernTCN

In [ ]:
RUN = True

modern_tcn_prediction_path = (
    BASELINE_CACHE_ROOT
    / "modern_tcn"
    / "prediction_result.pt"
)

if RUN:
    modern_tcn_checkpoint_path = Path(
        "/Users/vishalruparelia/Library/CloudStorage/"
        "GoogleDrive-vishal@autonomous-fox.ai/"
        "My Drive/dissertation/checkpoints/modern_tcn/"
        "per_asset_ohlcv_to_c_tod/runs/xff03ri9/"
        "best_checkpoint.pt"
    ).expanduser().resolve()

    modern_tcn = ModernTCNBaseline.from_config(
        config
    )

    modern_tcn.load_checkpoint(
        checkpoint_path=modern_tcn_checkpoint_path,
        device="cpu",
    )

    modern_tcn_result = modern_tcn.predict(
        split=test,
        batch_size=8,
        num_workers=0,
    )

    modern_tcn_prediction_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    torch.save(
        modern_tcn_result,
        modern_tcn_prediction_path,
    )

else:
    modern_tcn_result = torch.load(
        modern_tcn_prediction_path,
        map_location="cpu",
        weights_only=False,
    )

modern_tcn_evaluator = ForecastEvaluator(
    prediction_result=modern_tcn_result,
    train_split=train,
)

modern_tcn_results = modern_tcn_evaluator.evaluate(
    metrics=modern_tcn_evaluator.available_metrics,
    reduce_dims=(0, 2),
    bootstrap=True,
    n_bootstrap=BOOTSTRAP_N,
    confidence_level=BOOTSTRAP_CONFIDENCE_LEVEL,
    bootstrap_seed=BOOTSTRAP_SEED,
)

modern_tcn_metric_table = make_evaluation_table(
    metric_results=modern_tcn_results,
    horizons=modern_tcn_evaluator.horizons,
    channels=modern_tcn_evaluator.channels,
)

for metric_name in modern_tcn_evaluator.available_metrics:
    metric_display = (
        modern_tcn_metric_table
        .loc[
            modern_tcn_metric_table["metric"]
            == metric_name
        ]
        .set_index(
            [
                "horizon",
                "channel",
            ]
        )[
            [
                "value",
                "ci_lower",
                "ci_upper",
                "bootstrap_mean",
                "bootstrap_std",
            ]
        ]
    )

    display(
        metric_display.style.set_caption(
            metric_name
        )
    )

## Kronos

In [ ]:
kronos_result = torch.load(
    Path(
        "/Users/vishalruparelia/Library/CloudStorage/"
        "GoogleDrive-vishal@autonomous-fox.ai/"
        "My Drive/dissertation/kronos/"
        "kronos_small_test_fp16_bs16_progress.pt"
    ),
    map_location="cpu",
    weights_only=False,
)["prediction_result"]

kronos_evaluator = ForecastEvaluator(
    prediction_result=kronos_result,
    train_split=train,
)

kronos_results = kronos_evaluator.evaluate(
    metrics=kronos_evaluator.available_metrics,
    reduce_dims=(0, 2),
    bootstrap=True,
    n_bootstrap=BOOTSTRAP_N,
    confidence_level=BOOTSTRAP_CONFIDENCE_LEVEL,
    bootstrap_seed=BOOTSTRAP_SEED,
)

kronos_metric_table = make_evaluation_table(
    metric_results=kronos_results,
    horizons=kronos_evaluator.horizons,
    channels=kronos_evaluator.channels,
)

for metric_name in kronos_evaluator.available_metrics:
    display(
        kronos_metric_table
        .loc[kronos_metric_table["metric"] == metric_name]
        .set_index(["horizon", "channel"])[
            [
                "value",
                "ci_lower",
                "ci_upper",
                "bootstrap_mean",
                "bootstrap_std",
            ]
        ]
        .style.set_caption(metric_name)
    )